# Machine Translation Evaluation Pipeline

In [1]:
import pandas as pd
import os
import subprocess

# Ensure directories exist
os.makedirs("data/input", exist_ok=True)
os.makedirs("data/output", exist_ok=True)

In [ ]:
# Input Configuration
# Enter the path to your CSV file below
input_file_path = "data/input/Technical.csv"  # <-- Change this to your file path

if os.path.exists(input_file_path):
    print(f"Selected input file: {input_file_path}")
    df = pd.read_csv(input_file_path)
    print(f"Loaded {len(df)} rows.")
else:
    print(f"File not found: {input_file_path}")
    df = None

Selected input file: data/input/sample.csv
Loaded 2 rows.


In [3]:
# Run Pipeline
if df is not None:
    output_path = "data/output/evaluation_results.csv"
    
    # Determine executable
    exe = "mt_pipeline.exe" if os.name == 'nt' else "./mt_pipeline"
    if not os.path.exists(exe.replace("./", "")):
        if os.path.exists("mt_pipeline"):
            exe = "./mt_pipeline" if os.name != 'nt' else "mt_pipeline"
        elif os.path.exists("mt_pipeline.exe"):
            exe = "mt_pipeline.exe"
            
    print("Running pipeline...")
    try:
        result = subprocess.run([exe, input_file_path, output_path], capture_output=True, text=True, check=True)
        print(result.stdout)
        if result.stderr:
            print("Warnings:", result.stderr)
        print("Pipeline execution complete.")
    except Exception as e:
        print("Execution failed:", e)
        print("Ensure 'mt_pipeline' executable exists and is compiled.")
else:
    print("No valid input file loaded.")

Running pipeline...
[COMET] Loading model...

Fetching 5 files: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 5/5 [00:00<?, ?it/s]
[COMET] Using device: cpu
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\nikhi\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
d:\Research\Machine-Translation-Evaluation\venv\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
[COMET] Processing 2 entries...
ðŸ’¡ Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False

In [4]:
# View Results Row-by-Row
output_path = "data/output/evaluation_results.csv"

if os.path.exists(output_path):
    results = pd.read_csv(output_path)
    
    # Identify System Columns (any column that ends with _Bleu)
    system_names = set([c.rsplit('_', 1)[0] for c in results.columns if c.endswith('_Bleu')])
    
    print(f"Displaying {len(results)} Independent Entries:\n")
    
    for idx, row in results.iterrows():
        print(f"Entry #{idx + 1}")
        print("=" * 40)
        print(f"Source:    {row['Source']}")
        print(f"Reference: {row['Reference']}")
        print("-" * 40)
        
        for sys in sorted(list(system_names)):
            mt_text = row.get(sys, "N/A")
            bleu = row.get(f"{sys}_Bleu", 0)
            meteor = row.get(f"{sys}_Meteor", 0)
            comet = row.get(f"{sys}_Comet", 0)
            
            print(f"System: {sys}")
            print(f"  Text:   {mt_text}")
            print(f"  Scores: BLEU={bleu:.4f} | METEOR={meteor:.4f} | COMET={comet:.4f}")
            print("-" * 20)
        
        print("\n")
else:
    print("No results file found.")

Displaying 2 Independent Entries:

Entry #1
Source:    The sun rises in the east and sets in the west. Every morning brings new opportunities and challenges. We must embrace each day with hope and determination.
Reference: सूरज पूर्व में उगता है और पश्चिम में अस्त होता है। हर सुबह नए अवसर और चुनौतियाँ लाती है। हमें हर दिन को आशा और दृढ़ संकल्प के साथ स्वीकार करना चाहिए।
----------------------------------------
System: MT1
  Text:   सूरज पूर्व में उगता है और पश्चिम में अस्त होता है। हर सुबह नए अवसर और चुनौतियाँ लाती है। हमें हर दिन को आशा और दृढ़ संकल्प के साथ स्वीकार करना चाहिए।
  Scores: BLEU=1.0000 | METEOR=1.0000 | COMET=0.9499
--------------------
System: MT2
  Text:   सूर्य पूर्व में उगता है और पश्चिम में डूबता है। प्रत्येक सुबह नई संभावनाएं और कठिनाइयां लाती है। हमें प्रत्येक दिन को उम्मीद और दृढ़ता के साथ अपनाना चाहिए।
  Scores: BLEU=0.2697 | METEOR=0.7832 | COMET=0.8973
--------------------
System: MT3
  Text:   सूरज पूरब में निकलता है और पश्चिम में छिपता है। हर सुबह नए मौके और